In [1]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')

# Load feature matrix
features = pl.read_parquet("../data/feature_matrix.parquet")
print(f"Feature matrix: {features.shape}")
print(f"Late rate: {features['target_is_late'].mean():.1%}")

Feature matrix: (69963, 29)
Late rate: 25.4%


In [2]:
# Define feature groups
NUMERIC_FEATURES = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "is_morning_rush",
    "is_evening_rush",
    "raw_stop_sequence",
    "stop_position_bin",
    "uncertainty",
    "route_avg_delay",
    "route_stddev_delay",
    "route_late_rate",
    "stop_avg_delay",
    "stop_stddev_delay",
    "stop_late_rate",
    "temperature_f",
    "precipitation_mm",
    "wind_speed_mph",
    "visibility_m",
]

CATEGORICAL_FEATURES = [
    "route_id",
    "direction_id",
]

REGRESSION_TARGET = "target_delay_seconds"
CLASSIFICATION_TARGET = "target_is_late"

# Convert to pandas for sklearn
df = features.to_pandas()

# Encode categoricals
label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df[f"{col}_encoded"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

ENCODED_FEATURES = NUMERIC_FEATURES + [f"{c}_encoded" for c in CATEGORICAL_FEATURES]

# Drop rows with NaN in features
df_clean = df[ENCODED_FEATURES + [REGRESSION_TARGET, CLASSIFICATION_TARGET]].dropna()
print(f"Clean samples: {len(df_clean)} ({len(df_clean)/len(df):.1%} of total)")

X = df_clean[ENCODED_FEATURES].values
y_reg = df_clean[REGRESSION_TARGET].values
y_cls = df_clean[CLASSIFICATION_TARGET].values

# Split
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")
print(f"\nTrain late rate: {y_cls_train.mean():.1%}")
print(f"Test late rate:  {y_cls_test.mean():.1%}")

Clean samples: 69963 (100.0% of total)
Train: 55970 samples
Test:  13993 samples
Features: 20

Train late rate: 25.5%
Test late rate:  25.4%


In [3]:
# Baseline: predict mean delay / majority class
baseline_reg_pred = np.full_like(y_reg_test, y_reg_train.mean())
baseline_cls_pred = np.full_like(y_cls_test, 0)  # predict all on-time

print("=" * 60)
print("BASELINE MODELS")
print("=" * 60)
print(f"\nRegression Baseline (predict mean = {y_reg_train.mean():.1f}s):")
print(f"  MAE:  {mean_absolute_error(y_reg_test, baseline_reg_pred):.1f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_reg_test, baseline_reg_pred)):.1f}")
print(f"  R²:   {r2_score(y_reg_test, baseline_reg_pred):.4f}")

print("\nClassification Baseline (predict all on-time):")
print(f"  Accuracy:  {accuracy_score(y_cls_test, baseline_cls_pred):.4f}")
print(f"  Precision: {precision_score(y_cls_test, baseline_cls_pred, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_cls_test, baseline_cls_pred):.4f}")
print(f"  F1:        {f1_score(y_cls_test, baseline_cls_pred):.4f}")

BASELINE MODELS

Regression Baseline (predict mean = -46.0s):
  MAE:  235.0
  RMSE: 323.5
  R²:   -0.0001

Classification Baseline (predict all on-time):
  Accuracy:  0.7463
  Precision: 0.0000
  Recall:    0.0000
  F1:        0.0000


In [4]:
# Train regression models
reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge (α=1.0)": Ridge(alpha=1.0),
    "Ridge (α=10.0)": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_leaf=10, random_state=42, n_jobs=-1
    ),
}

reg_results = []

for name, model in reg_models.items():
    # Use scaled for linear, unscaled for tree
    if "Forest" in name:
        model.fit(X_train, y_reg_train)
        preds = model.predict(X_test)
    else:
        model.fit(X_train_scaled, y_reg_train)
        preds = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_reg_test, preds)
    rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
    r2 = r2_score(y_reg_test, preds)

    reg_results.append({
        "model": name,
        "MAE": round(mae, 1),
        "RMSE": round(rmse, 1),
        "R²": round(r2, 4),
        "predictions": preds,
    })

    print(f"{name:<25} MAE={mae:.1f}  RMSE={rmse:.1f}  R²={r2:.4f}")

# Add baseline
reg_results.insert(0, {
    "model": "Baseline (mean)",
    "MAE": round(mean_absolute_error(y_reg_test, baseline_reg_pred), 1),
    "RMSE": round(np.sqrt(mean_squared_error(y_reg_test, baseline_reg_pred)), 1),
    "R²": round(r2_score(y_reg_test, baseline_reg_pred), 4),
})

print("\n" + "=" * 60)
reg_comparison = pd.DataFrame(reg_results).drop(columns=["predictions"], errors="ignore")
print(reg_comparison.to_string(index=False))

Linear Regression         MAE=125.7  RMSE=191.7  R²=0.6490
Ridge (α=1.0)             MAE=125.7  RMSE=191.7  R²=0.6490
Ridge (α=10.0)            MAE=125.7  RMSE=191.7  R²=0.6490
Random Forest             MAE=81.6  RMSE=145.9  R²=0.7966

            model   MAE  RMSE      R²
  Baseline (mean) 235.0 323.5 -0.0001
Linear Regression 125.7 191.7  0.6490
    Ridge (α=1.0) 125.7 191.7  0.6490
   Ridge (α=10.0) 125.7 191.7  0.6490
    Random Forest  81.6 145.9  0.7966


In [5]:
reg_df = pd.DataFrame(reg_results).drop(columns=["predictions"], errors="ignore")

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("MAE (lower = better)", "RMSE (lower = better)", "R² (higher = better)")
)

fig.add_trace(go.Bar(x=reg_df["model"], y=reg_df["MAE"], marker_color="#3498db", text=reg_df["MAE"], textposition="outside"), row=1, col=1)
fig.add_trace(go.Bar(x=reg_df["model"], y=reg_df["RMSE"], marker_color="#e74c3c", text=reg_df["RMSE"], textposition="outside"), row=1, col=2)
fig.add_trace(go.Bar(x=reg_df["model"], y=reg_df["R²"], marker_color="#2ecc71", text=reg_df["R²"], textposition="outside"), row=1, col=3)

fig.update_layout(height=450, title_text="Regression Model Comparison", showlegend=False)
fig.update_xaxes(tickangle=-30)
fig.show()

In [6]:
# Use Random Forest as best model (typically)
best_reg_name = max(reg_results[1:], key=lambda x: x["R²"])["model"]
best_reg_preds = [r for r in reg_results if r["model"] == best_reg_name][0]["predictions"]

residuals = y_reg_test - best_reg_preds

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Predicted vs Actual", "Residual Distribution")
)

# Sample for scatter to avoid overplotting
sample_idx = np.random.choice(len(y_reg_test), min(3000, len(y_reg_test)), replace=False)

fig.add_trace(
    go.Scatter(
        x=y_reg_test[sample_idx],
        y=best_reg_preds[sample_idx],
        mode="markers",
        marker=dict(size=3, opacity=0.3, color="#3498db"),
        name="Predictions",
    ),
    row=1, col=1,
)

# Perfect prediction line
min_val = min(y_reg_test.min(), best_reg_preds.min())
max_val = max(y_reg_test.max(), best_reg_preds.max())
fig.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode="lines", line=dict(color="red", dash="dash"), name="Perfect"),
    row=1, col=1,
)

fig.add_trace(
    go.Histogram(x=residuals, nbinsx=80, marker_color="#2ecc71", name="Residuals"),
    row=1, col=2,
)

fig.update_layout(height=450, title_text=f"Best Regression Model: {best_reg_name}", showlegend=False)
fig.update_xaxes(title_text="Actual Delay (s)", row=1, col=1)
fig.update_yaxes(title_text="Predicted Delay (s)", row=1, col=1)
fig.update_xaxes(title_text="Residual (s)", row=1, col=2)
fig.show()

print(f"\nResidual Statistics ({best_reg_name}):")
print(f"  Mean:   {residuals.mean():.1f}s")
print(f"  Std:    {residuals.std():.1f}s")
print(f"  Median: {np.median(residuals):.1f}s")


Residual Statistics (Random Forest):
  Mean:   -0.2s
  Std:    145.9s
  Median: 0.0s


In [7]:
# Train classification models
cls_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_leaf=10, random_state=42, n_jobs=-1
    ),
}

cls_results = []

for name, model in cls_models.items():
    if "Forest" in name:
        model.fit(X_train, y_cls_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
    else:
        model.fit(X_train_scaled, y_cls_train)
        preds = model.predict(X_test_scaled)
        proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_cls_test, preds)
    prec = precision_score(y_cls_test, preds)
    rec = recall_score(y_cls_test, preds)
    f1 = f1_score(y_cls_test, preds)
    auc = roc_auc_score(y_cls_test, proba)

    cls_results.append({
        "model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "AUC": round(auc, 4),
        "predictions": preds,
        "probabilities": proba,
    })

    print(f"\n{name}")
    print(f"  Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f}")

# Add baseline
cls_results.insert(0, {
    "model": "Baseline (all on-time)",
    "Accuracy": round(accuracy_score(y_cls_test, baseline_cls_pred), 4),
    "Precision": 0.0,
    "Recall": 0.0,
    "F1": 0.0,
    "AUC": 0.5,
})

print("\n" + "=" * 60)
cls_comparison = pd.DataFrame(cls_results).drop(columns=["predictions", "probabilities"], errors="ignore")
print(cls_comparison.to_string(index=False))


Logistic Regression
  Accuracy=0.8617  Precision=0.7746  Recall=0.6417  F1=0.7019  AUC=0.9280

Random Forest
  Accuracy=0.9083  Precision=0.8651  Recall=0.7566  F1=0.8072  AUC=0.9700

                 model  Accuracy  Precision  Recall     F1   AUC
Baseline (all on-time)    0.7463     0.0000  0.0000 0.0000 0.500
   Logistic Regression    0.8617     0.7746  0.6417 0.7019 0.928
         Random Forest    0.9083     0.8651  0.7566 0.8072 0.970


In [8]:
fig = go.Figure()

for result in cls_results[1:]:  # skip baseline
    if "probabilities" in result:
        fpr, tpr, _ = roc_curve(y_cls_test, result["probabilities"])
        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            mode="lines",
            name=f"{result['model']} (AUC={result['AUC']})",
        ))

# Random baseline
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    line=dict(dash="dash", color="gray"),
    name="Random (AUC=0.5)",
))

fig.update_layout(
    title="ROC Curves — Late Prediction",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=500,
)
fig.show()

In [9]:
# Use best F1 model
best_cls = max(cls_results[1:], key=lambda x: x["F1"])
best_cls_preds = best_cls["predictions"]

cm = confusion_matrix(y_cls_test, best_cls_preds)

fig = px.imshow(
    cm,
    text_auto=True,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=["On Time", "Late"],
    y=["On Time", "Late"],
    title=f"Confusion Matrix — {best_cls['model']}",
    color_continuous_scale="Blues",
)
fig.update_layout(height=450)
fig.show()

print(f"\nClassification Report — {best_cls['model']}:")
print(classification_report(y_cls_test, best_cls_preds, target_names=["On Time", "Late"]))


Classification Report — Random Forest:
              precision    recall  f1-score   support

     On Time       0.92      0.96      0.94     10443
        Late       0.87      0.76      0.81      3550

    accuracy                           0.91     13993
   macro avg       0.89      0.86      0.87     13993
weighted avg       0.91      0.91      0.91     13993



In [10]:
# Get feature importance from Random Forest (regression)
rf_reg = [m for name, m in reg_models.items() if "Random Forest" in name][0]

importance_df = pd.DataFrame({
    "feature": ENCODED_FEATURES,
    "importance": rf_reg.feature_importances_
}).sort_values("importance", ascending=True)

fig = px.bar(
    importance_df.tail(15),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 15 Feature Importances (Random Forest Regression)",
    labels={"importance": "Importance", "feature": "Feature"},
    color="importance",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=500)
fig.show()

# Also for classifier
rf_cls = [m for name, m in cls_models.items() if "Random Forest" in name][0]

cls_importance_df = pd.DataFrame({
    "feature": ENCODED_FEATURES,
    "importance": rf_cls.feature_importances_
}).sort_values("importance", ascending=True)

fig = px.bar(
    cls_importance_df.tail(15),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 15 Feature Importances (Random Forest Classifier)",
    color="importance",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=500)
fig.show()

In [11]:
# Cross-validate best models
print("Cross-Validation (5-fold)")
print("=" * 50)

# Regression CV
rf_reg_cv = cross_val_score(
    RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=10, random_state=42, n_jobs=-1),
    X, y_reg, cv=5, scoring="neg_mean_absolute_error"
)
print("\nRandom Forest Regression — MAE:")
print(f"  Folds: {[-round(s, 1) for s in rf_reg_cv]}")
print(f"  Mean:  {-rf_reg_cv.mean():.1f} ± {rf_reg_cv.std():.1f}")

ridge_cv = cross_val_score(
    Ridge(alpha=1.0),
    scaler.fit_transform(X), y_reg, cv=5, scoring="neg_mean_absolute_error"
)
print("\nRidge Regression — MAE:")
print(f"  Folds: {[-round(s, 1) for s in ridge_cv]}")
print(f"  Mean:  {-ridge_cv.mean():.1f} ± {ridge_cv.std():.1f}")

# Classification CV
rf_cls_cv = cross_val_score(
    RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=10, random_state=42, n_jobs=-1),
    X, y_cls, cv=5, scoring="f1"
)
print("\nRandom Forest Classifier — F1:")
print(f"  Folds: {[round(s, 4) for s in rf_cls_cv]}")
print(f"  Mean:  {rf_cls_cv.mean():.4f} ± {rf_cls_cv.std():.4f}")

Cross-Validation (5-fold)

Random Forest Regression — MAE:
  Folds: [244.2, 216.5, 60.5, 39.0, 120.8]
  Mean:  136.2 ± 81.9

Ridge Regression — MAE:
  Folds: [246.7, 213.0, 103.3, 61.4, 119.6]
  Mean:  148.8 ± 69.7

Random Forest Classifier — F1:
  Folds: [0.4947, 0.5152, 0.7535, 0.071, 0.8544]
  Mean:  0.5378 ± 0.2710


In [12]:
print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)

print("\nREGRESSION (Predict delay in seconds)")
print(reg_comparison.to_string(index=False))

print("\nCLASSIFICATION (Predict late vs on-time)")
print(cls_comparison.to_string(index=False))

best_reg = max(reg_results[1:], key=lambda x: x["R²"])
best_cls = max(cls_results[1:], key=lambda x: x["F1"])

print(f"""
BEST MODELS:
  Regression:     {best_reg['model']} (R²={best_reg['R²']}, MAE={best_reg['MAE']}s)
  Classification: {best_cls['model']} (F1={best_cls['F1']}, AUC={best_cls['AUC']})

IMPROVEMENT OVER BASELINE:
  Regression MAE:  {reg_results[0]['MAE']}s → {best_reg['MAE']}s ({(1 - best_reg['MAE']/reg_results[0]['MAE'])*100:.1f}% improvement)
  Classification F1: 0.0 → {best_cls['F1']} 

KEY INSIGHTS:
  - Top features: stop/route historical delays dominate
  - Temporal features matter less with single-day data
  - Weather features will improve with multi-day collection
  - More data (days/weeks) will significantly improve all models

NEXT: Notebook 05 — Detailed model evaluation and error analysis
""")

MODEL TRAINING SUMMARY

REGRESSION (Predict delay in seconds)
            model   MAE  RMSE      R²
  Baseline (mean) 235.0 323.5 -0.0001
Linear Regression 125.7 191.7  0.6490
    Ridge (α=1.0) 125.7 191.7  0.6490
   Ridge (α=10.0) 125.7 191.7  0.6490
    Random Forest  81.6 145.9  0.7966

CLASSIFICATION (Predict late vs on-time)
                 model  Accuracy  Precision  Recall     F1   AUC
Baseline (all on-time)    0.7463     0.0000  0.0000 0.0000 0.500
   Logistic Regression    0.8617     0.7746  0.6417 0.7019 0.928
         Random Forest    0.9083     0.8651  0.7566 0.8072 0.970

BEST MODELS:
  Regression:     Random Forest (R²=0.7966, MAE=81.6s)
  Classification: Random Forest (F1=0.8072, AUC=0.97)

IMPROVEMENT OVER BASELINE:
  Regression MAE:  235.0s → 81.6s (65.3% improvement)
  Classification F1: 0.0 → 0.8072 

KEY INSIGHTS:
  - Top features: stop/route historical delays dominate
  - Temporal features matter less with single-day data
  - Weather features will improve with mul